# 🔍 Test GraphRAG Search Functions

Notebook này dùng để kiểm tra xem các hàm search hoạt động đúng không:
- `local_search_hybrid` — tìm kiếm entities/relationships trong knowledge graph
- `global_search_hybrid` — tìm kiếm community summaries (tổng quan)
- `naive_search` — tìm kiếm theo cosine similarity thuần (FAISS)


## 1. Setup — import và warmup

In [ ]:
import os
import sys

# Đảm bảo chạy từ thư mục backend
os.chdir(os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd())
print("Working dir:", os.getcwd())

from graphrag_tools import local_search_hybrid, global_search_hybrid, naive_search, warmup
print("✅ Import thành công")

In [ ]:
# Warmup: load graph cache + FAISS index một lần duy nhất
warmup()

---
## 2. Test `local_search_hybrid`

Hàm này tìm entities + relationships trong knowledge graph, phù hợp cho câu hỏi **cụ thể, chi tiết**.

In [ ]:
def test_local_search(query: str, top_k: int = 10):
    """Test local_search_hybrid với một câu hỏi."""
    print(f"\n{'='*60}")
    print(f"📍 LOCAL SEARCH | query: '{query}' | top_k={top_k}")
    print('='*60)
    try:
        result = local_search_hybrid(query, top_k=top_k)
        print(result)
        print(f"\n✅ Độ dài kết quả: {len(result)} ký tự")
    except Exception as e:
        print(f"❌ Lỗi: {e}")
        raise

In [ ]:
# Test 1: câu hỏi về điều kiện tốt nghiệp
test_local_search("điều kiện tốt nghiệp đại học")

In [ ]:
# Test 2: câu hỏi về điểm số, xếp loại
test_local_search("điểm trung bình tích lũy xếp loại học lực")

In [ ]:
# Test 3: câu hỏi về học phần, tín chỉ
test_local_search("đăng ký học phần tín chỉ")

---
## 3. Test `global_search_hybrid`

Hàm này tìm community summaries (báo cáo tổng quan), phù hợp cho câu hỏi **tổng quan, so sánh nhiều chủ đề**.

In [ ]:
def test_global_search(query: str, top_k: int = 5):
    """Test global_search_hybrid với một câu hỏi."""
    print(f"\n{'='*60}")
    print(f"🌐 GLOBAL SEARCH | query: '{query}' | top_k={top_k}")
    print('='*60)
    try:
        result = global_search_hybrid(query, top_k=top_k)
        print(result)
        print(f"\n✅ Độ dài kết quả: {len(result)} ký tự")
    except Exception as e:
        print(f"❌ Lỗi: {e}")
        raise

In [ ]:
# Test 1: tổng quan quy chế đào tạo
test_global_search("tổng quan quy chế đào tạo đại học HUST")

In [ ]:
# Test 2: chính sách học bổng, khen thưởng
test_global_search("chính sách học bổng khen thưởng sinh viên")

In [ ]:
# Test 3: xử lý kỷ luật sinh viên
test_global_search("kỷ luật sinh viên vi phạm quy chế")

---
## 4. Test `naive_search`

Hàm này dùng FAISS cosine similarity trực tiếp trên raw text chunks, không dùng graph.

In [ ]:
def test_naive_search(query: str, top_k: int = 5):
    """Test naive_search với một câu hỏi."""
    print(f"\n{'='*60}")
    print(f"🔎 NAIVE SEARCH | query: '{query}' | top_k={top_k}")
    print('='*60)
    try:
        result = naive_search(query, top_k=top_k)
        print(result)
        print(f"\n✅ Độ dài kết quả: {len(result)} ký tự")
    except Exception as e:
        print(f"❌ Lỗi: {e}")
        raise

In [ ]:
test_naive_search("điều kiện tốt nghiệp")

In [ ]:
test_naive_search("quy định về thi lại, học lại")

---
## 5. So sánh Local vs Global vs Naive

Chạy cùng một câu hỏi qua cả 3 hàm để so sánh.

In [ ]:
def compare_all(query: str):
    """Chạy cùng câu hỏi qua cả 3 search, in kết quả và thống kê."""
    import time

    print(f"\n{'#'*70}")
    print(f"  SO SÁNH 3 PHƯƠNG THỨC SEARCH")
    print(f"  Query: '{query}'")
    print(f"{'#'*70}")

    results = {}
    for name, fn, kwargs in [
        ("local_search_hybrid",  local_search_hybrid,  {"top_k": 10}),
        ("global_search_hybrid", global_search_hybrid, {"top_k": 5}),
        ("naive_search",         naive_search,         {"top_k": 5}),
    ]:
        t0 = time.time()
        try:
            out = fn(query, **kwargs)
            elapsed = time.time() - t0
            results[name] = {"output": out, "time": elapsed, "error": None}
        except Exception as e:
            elapsed = time.time() - t0
            results[name] = {"output": "", "time": elapsed, "error": str(e)}

    # In bảng thống kê
    print(f"\n{'─'*60}")
    print(f"{'Hàm':<25} {'Thời gian':>10} {'Độ dài':>10} {'Trạng thái':>12}")
    print(f"{'─'*60}")
    for name, r in results.items():
        status = "✅ OK" if r["error"] is None else "❌ LỖI"
        print(f"{name:<25} {r['time']:>9.2f}s {len(r['output']):>10} {status:>12}")
    print(f"{'─'*60}")

    # In chi tiết từng kết quả
    for name, r in results.items():
        print(f"\n{'='*60}")
        print(f"📄 {name}")
        print('='*60)
        if r["error"]:
            print(f"❌ Lỗi: {r['error']}")
        else:
            # Chỉ in 800 ký tự đầu để không quá dài
            preview = r["output"][:800]
            print(preview)
            if len(r["output"]) > 800:
                print(f"\n... (còn {len(r['output'])-800} ký tự nữa)")

In [ ]:
compare_all("điều kiện xét tốt nghiệp đại học bách khoa")

In [ ]:
compare_all("quy định thi lại học lại môn học")

---
## 6. Test nhanh — chạy 1 dòng

In [ ]:
# Thay câu hỏi tùy ý ở đây và chạy cell này
MY_QUERY = "sinh viên bị đình chỉ học tập khi nào"

print("--- LOCAL ---")
print(local_search_hybrid(MY_QUERY)[:600])

print("\n--- GLOBAL ---")
print(global_search_hybrid(MY_QUERY)[:600])